# Exercise 8: Abstention in language identification

*Scripts and Encoding · version 0.1.0*

> Build a sample of 100 short texts: 60 in Turkic languages, 20 in Russian, 10 in English and 10 mixed Kazakh–Russian. Run `LanguageDetection()` restricted to Turkic labels and the unrestricted model. For a range of thresholds, report precision for each Turkic language and the share of non-Turkic texts that are accepted. Which threshold and abstention rule would you choose for a crawl, and why?

The sample is built from X-WMT test sentences: 10 each in Turkish, Azerbaijani, Kazakh, Kyrgyz, Uzbek and Bashkir, 20 Russian, 10 English, and 10 synthetic Kazakh–Russian mixtures (the first half of a Kazakh sentence joined to the second half of a Russian one). `LanguageDetection()` downloads GlotLID on first use and needs `fasttext`.

**Setup.** `pip install "turkicnlp==0.5.0"[lid] requests pandas`. Cells that fetch data from the web need internet access.

In [ ]:
import requests

BASE = "https://raw.githubusercontent.com/turkic-interlingua/til-mt/master/xwmt/test"

def xwmt_side(pair, lang):
    """One side of an X-WMT test pair, e.g. xwmt_side("en-ky", "ky")."""
    r = requests.get(f"{BASE}/{pair}/{pair}.{lang}", timeout=30)
    r.raise_for_status()
    return r.text.splitlines()

In [ ]:
import pandas as pd
import turkicnlp

GOLD = {"tr": "tur_Latn", "az": "azj_Latn", "kk": "kaz_Cyrl", "ky": "kir_Cyrl", "uz": "uzn_Latn", "ba": "bak_Cyrl"}
sample = []
for code, label in GOLD.items():
    sample += [(s, label) for s in xwmt_side(f"en-{code}", code)[:10]]
sample += [(s, "rus_Cyrl") for s in xwmt_side("en-ru", "ru")[:20]]
sample += [(s, "eng_Latn") for s in xwmt_side("en-kk", "en")[100:110]]
kk, ru = xwmt_side("ru-kk", "kk")[200:210], xwmt_side("ru-kk", "ru")[200:210]
sample += [(" ".join(a.split()[: len(a.split()) // 2] + b.split()[len(b.split()) // 2:]), "mixed") for a, b in zip(kk, ru)]
print(len(sample), "texts")

In [ ]:
lid = turkicnlp.LanguageDetection()                 # restricted to Turkic labels
turkic = {l.replace("__label__", "") for l in lid.available_labels()}

rows = []
for text, gold in sample:
    full_labels, full_scores = lid.model.predict(text.replace("\n", " "), k=2)   # unrestricted GlotLID
    r_labels, r_scores = lid.predict(text, k=1)
    rows.append({"gold": gold, "full": full_labels[0].replace("__label__", ""), "full_score": float(full_scores[0]),
                 "margin": float(full_scores[0] - (full_scores[1] if len(full_scores) > 1 else 0)),
                 "restricted": r_labels[0].replace("__label__", "")})
df = pd.DataFrame(rows)
df.head()

In [ ]:
def evaluate(min_score, min_margin):
    accept = df["full"].isin(turkic) & (df["full_score"] >= min_score) & (df["margin"] >= min_margin)
    out = {"min_score": min_score, "min_margin": min_margin,
           "non-Turkic accepted": f"{(accept & ~df['gold'].isin(GOLD.values())).sum()}/{(~df['gold'].isin(GOLD.values())).sum()}"}
    for label in GOLD.values():
        predicted = accept & (df["restricted"] == label)
        out[label] = round((df.loc[predicted, "gold"] == label).mean(), 2) if predicted.any() else None
    return out

print("Restricted model without abstention: non-Turkic texts forced into a Turkic label:",
      (~df["gold"].isin(GOLD.values())).sum())
pd.DataFrame([evaluate(s, m) for s in (0.0, 0.5, 0.7, 0.9) for m in (0.0, 0.3)])

## Worked solution

This experiment needs the GlotLID model, which could not be downloaded in the environment where these notebooks were prepared, so no scores are recorded here; run the cells to obtain yours.

**What the comparison shows.** The restricted model has no way to say "not Turkic": every Russian, English and mixed text receives some Turkic label, so its acceptance rate for non-Turkic text is 100%. The unrestricted model with a Turkic-family check, a minimum score and a minimum margin between the top two labels can reject them, at the cost of rejecting some short or ambiguous Turkic texts.

**Choosing a rule for a crawl.** Accept a text only if (1) the unrestricted top label is Turkic, (2) its score and margin exceed thresholds chosen on labelled calibration data from the target crawl, and (3) the restricted label agrees; send everything else to an "unknown" bin for inspection. Choose the thresholds to keep the accepted non-Turkic share near zero, then report the Turkic recall you lose. The 100 texts here are too few to fix thresholds: use them to understand the trade-off, and calibrate on a larger sample that includes short strings, names, boilerplate and real code-switching (the synthetic mixtures here are not natural code-switched text).